# Train a quantum CartPole policy

Map a four-component CartPole observation to action probabilities with a two-qubit policy. Train with REINFORCE and inspect a concrete action distribution.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)
figures = Path("figures")
figures.mkdir(exist_ok=True)

## Configure the experiment

A NeuralEmbedding learns one four-component projection per wire. The quantum expectations feed a classical action head. The init and input_scaling_init fields initialize the quantum angles and projection weights.

In [ ]:
from qasp import Normal, Uniform
from copy import deepcopy
from qasp import create_model, draw_model, REINFORCE

np.random.seed(2026)
torch.manual_seed(2026)
n_qubits = 2
policy_config = {
    "feature_map": "NeuralEmbedding",
    "feature_map_config": {"features": 4, "rotations": "Y"},
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 1, "rotations": "YZ",
        "include_imprimitive": True,
        "imprimitive": "CZ", "imprimitive_pattern": "chain",
    },
    "n_actions": 2,
    "observables": ["Z 0", "Z 1"],
    "fc_layer": True,
    "init": Normal(0, 0.1),
    "input_scaling_init": Uniform(0.8, 1.2),
    "device": "default.qubit",
    "diff_method": "backprop",
    "shots": None,
}
trainer_config = {
    "env_name": "CartPole-v1", "device": "cpu",
    "lr": 0.01, "gamma": 0.99,
    "max_steps": 8, "batch_episodes": 2,
    "logits": True, "normalize_mode": "none",
    "baseline_type": "ema", "entropy_coef": 0.01,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
policy = create_model(
    "PQCSoftmax", deepcopy(policy_config), n_qubits=n_qubits
)
probe = torch.tensor([0.12, -0.04, 0.08, 0.02], dtype=torch.float32)
logits = policy(probe)
assert logits.shape == (2,)
print("Action probabilities:", torch.softmax(logits, dim=-1).detach())
print("Initial angles:", policy.qlayer.params.detach())
print("Initial encoding weights:", policy.qlayer.input_scaling_params.detach())

## Train

REINFORCE estimates a policy gradient from sampled actions and episode returns. The short episode and training budgets keep the example quick. Increasing those budgets is necessary for a learning-performance study.

In [ ]:
agent = REINFORCE(policy=policy, **trainer_config)
agent.env.reset(seed=2026)
try:
    training_returns = agent.train(num_episodes=4, log_every=2)
finally:
    agent.env.close()

assert len(training_returns) == 4
assert np.isfinite(training_returns).all()

## Read the results

Inspect episode returns and the distribution for the original observation. Argmax is the greedy decision; during training the agent samples from the distribution.

In [ ]:
policy.eval()
with torch.no_grad():
    action_logits = policy(probe)
    action_probabilities = torch.softmax(action_logits, dim=-1)
    greedy_action = int(action_probabilities.argmax().item())

print("Episode returns:", training_returns)
print("Mean training return:", float(np.mean(training_returns)))
print("Probabilities:", action_probabilities)
print("Greedy action:", greedy_action)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(np.arange(len(training_returns))+1, training_returns)
ax.set(xlabel="Training episode", ylabel="Episode return")
plt.show()

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_model(model=policy, inputs=probe)
display(fig)
plt.close(fig)

### Export a paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

In [ ]:
import shutil
import subprocess

def show_tikz(source, name):
    """Write standalone Quantikz and, when available, render its PDF inline."""
    tex = figures / f"{name}.tex"
    tex.write_text(source, encoding="utf-8")
    print("LaTeX:", tex)
    if shutil.which("pdflatex") is None:
        print("Install a TeX distribution with Quantikz to compile the PDF.")
        return
    subprocess.run(["pdflatex", "-interaction=batchmode", "-halt-on-error", tex.name],
                   cwd=figures, check=True)
    pdf = tex.with_suffix(".pdf")
    print("Vector PDF:", pdf)
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if gs:
        png = subprocess.check_output([gs, "-q", "-sDEVICE=png16m", "-r120",
                                      "-o", "-", pdf.name], cwd=figures)
        display(Image(data=png))

In [ ]:
tex = draw_model(
    model=policy, inputs=probe, output="tikz",
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)
show_tikz(tex, "policy")

## Try it

Change the projection-weight distribution, reconstruct the policy and compare action probabilities at the same nonzero observation.